# 23 - Gradient Boosting-ARX

Este notebook entrena y evalua un modelo **Gradient Boosting-ARX** para predecir el nivel del Mburicao a 10, 20 y 30 minutos.

La metodologia replica el protocolo planteado para los modelos finales:

- segmentos 2021 y 2025 tratados por separado;
- frecuencia regular de 10 minutos;
- split temporal 70% train, 15% validation, 15% test;
- modelo global entrenado con ejemplos de ambos segmentos;
- features ARX construidas solo con informacion historica;
- grid search de hiperparametros usando validation;
- evaluacion final sobre test;
- tablas de metricas globales, por segmento, por lead time y por condiciones criticas.

Este notebook **no compara aun contra otros modelos**. Solo deja los resultados listos, con el mismo formato de reportes, para una comparacion posterior.

## Rol metodologico del Gradient Boosting-ARX

Ridge-ARX evalua si una combinacion lineal de variables historicas de nivel y lluvia es suficiente para predecir el nivel futuro. Gradient Boosting-ARX usa la misma representacion ARX, pero reemplaza la relacion lineal por un ensamble no lineal de arboles.

Esto permite evaluar una pregunta intermedia entre Ridge-ARX y deep learning:

```text
Hay no linealidades tabulares explotables en las features ARX?
```

Si Gradient Boosting-ARX mejora claramente sobre Ridge-ARX, la mejora puede deberse a interacciones no lineales entre nivel, lluvia acumulada y cambios recientes. Si no mejora, puede indicar que las features ARX ya capturan principalmente una dinamica local casi lineal, o que hacen falta representaciones secuenciales mas flexibles.

## Que significa ARX en este notebook

ARX significa **AutoRegressive with eXogenous inputs**. En este contexto, el modelo usa informacion autoregresiva del propio nivel del cauce y variables exogenas relacionadas con la precipitacion.

La parte autoregresiva corresponde a variables derivadas del nivel:

```text
nivel actual
cambios recientes del nivel
medias moviles recientes del nivel
```

La parte exogena corresponde a variables derivadas de lluvia:

```text
lluvia reciente
lluvia acumulada en ventanas cortas
maximos recientes de precipitacion
```

La idea de ARX es transformar un problema de serie temporal en un problema supervisado tabular. En lugar de entregar al modelo una serie completa, se construye una fila por cada origen de prediccion:

```text
X_i = resumen historico disponible hasta el origen t
y_i = nivel futuro en t+10, t+20 y t+30 minutos
```

Esto permite entrenar modelos de machine learning tabulares, como Ridge o Gradient Boosting, sobre un problema originalmente temporal.

Una condicion metodologica importante es que todas las variables de `X_i` se calculan usando solo informacion historica disponible antes del horizonte futuro. El modelo no recibe lluvia futura ni nivel futuro dentro de las variables predictoras.

## Como funciona Gradient Boosting

Gradient Boosting es un metodo de ensamble secuencial de arboles de decision. En lugar de entrenar un unico arbol grande, entrena muchos arboles pequenos de forma aditiva.

La intuicion es:

```text
modelo_0 predice de forma simple
arbol_1 aprende los errores de modelo_0
arbol_2 aprende los errores que aun quedan
arbol_3 corrige errores residuales
...
prediccion final = suma ponderada de muchos arboles
```

Cada arbol intenta reducir el error residual del ensamble anterior. Por eso se llama *boosting*: cada modelo nuevo refuerza o corrige al conjunto previo.

En regresion, el modelo aprende una funcion no lineal:

```text
f(X) -> y
```

donde `X` son las features ARX y `y` es el nivel futuro. A diferencia de Ridge-ARX, que impone una relacion lineal, Gradient Boosting puede capturar:

- umbrales;
- saturaciones;
- interacciones entre variables;
- respuestas distintas segun el estado del cauce;
- relaciones no lineales entre lluvia reciente y ascenso del nivel.

Por ejemplo, puede aprender reglas del tipo:

```text
si el nivel ya esta alto
y la lluvia acumulada en 60 minutos es elevada
y el delta reciente del nivel es positivo
entonces el aumento futuro esperado es mayor.
```

Este tipo de regla es dificil de representar con un modelo lineal simple, salvo que se construyan manualmente muchas interacciones.

## Entrada y salida del modelo

Este notebook entrena un modelo multi-horizonte. Cada fila de entrada contiene features historicas ARX, y cada salida contiene tres valores futuros del nivel.

La entrada tiene forma:

```text
X.shape = [numero_de_ejemplos, numero_de_features]
```

En este notebook:

```text
numero_de_features = 18
```

La salida tiene forma:

```text
y.shape = [numero_de_ejemplos, 3]
```

Los tres valores de salida son:

```text
y[:, 0] = Nivel(t+10)
y[:, 1] = Nivel(t+20)
y[:, 2] = Nivel(t+30)
```

Como `GradientBoostingRegressor` de scikit-learn predice una sola salida por modelo, se usa `MultiOutputRegressor`. Esto entrena internamente un modelo por horizonte:

```text
GradientBoosting para t+10
GradientBoosting para t+20
GradientBoosting para t+30
```

Los tres modelos reciben las mismas variables de entrada, pero cada uno aprende a predecir un horizonte diferente. En terminos practicos, el objeto final se comporta como un unico modelo multi-salida:

```text
X_i -> [Nivel(t+10), Nivel(t+20), Nivel(t+30)]
```

Esta estructura es comparable con Ridge-ARX, N-BEATS y LSTM porque todos producen predicciones para los mismos tres horizontes.

## Imports

In [1]:
import os
import time
import warnings
from itertools import product

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## Configuracion experimental

In [2]:
TIME_COL = "Time"
TARGET_COL = "Nivel"
RAIN_COL = "Precipitacion"

FREQ = "10min"
STEPS_PER_HOUR = 6
FORECAST_HORIZON = 3
EVAL_STRIDE = 1

TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 1.0 - TRAIN_FRACTION - VAL_FRACTION

RANDOM_STATE = 42
MODEL_NAME = "GradientBoosting_ARX"

# Se mantiene False para replicar el protocolo usado en los modelos finales:
# validation selecciona hiperparametros, test queda reservado para metricas finales.
FINAL_TRAIN_USES_VALIDATION = False

print("Split:")
print(f"Train: {TRAIN_FRACTION:.0%}")
print(f"Validation: {VAL_FRACTION:.0%}")
print(f"Test: {TEST_FRACTION:.0%}")

print("\nLead times evaluados:")
for step in range(1, FORECAST_HORIZON + 1):
    print(f"lead_step={step}: t+{step * 10} min")

Split:
Train: 70%
Validation: 15%
Test: 15%

Lead times evaluados:
lead_step=1: t+10 min
lead_step=2: t+20 min
lead_step=3: t+30 min


## Grid search

## Por que se usa grid search

Gradient Boosting tiene hiperparametros que controlan la complejidad del ensamble. No conviene elegirlos arbitrariamente, porque una configuracion demasiado simple puede subajustar, mientras que una configuracion demasiado compleja puede sobreajustar.

El grid search prueba combinaciones predefinidas y selecciona la mejor usando el conjunto de validation. El test queda reservado para la evaluacion final.

Los hiperparametros evaluados son:

- `n_estimators`: cantidad de arboles del ensamble. Mas arboles aumentan capacidad, pero tambien costo y riesgo de sobreajuste.
- `learning_rate`: peso con que se incorpora cada arbol nuevo. Valores bajos hacen el aprendizaje mas gradual.
- `max_depth`: profundidad maxima de cada arbol. Controla la complejidad de las interacciones aprendidas.
- `min_samples_leaf`: minimo de muestras por hoja. Valores mayores regularizan el modelo.
- `subsample`: fraccion de muestras usada por cada arbol. Si es menor que 1, introduce aleatoriedad y puede mejorar generalizacion.

La seleccion se realiza segun `RMSE` global en validation, porque RMSE penaliza con fuerza errores grandes, relevantes en eventos de crecida.

In [3]:
GB_GRID = []

for n_estimators, learning_rate, max_depth, min_samples_leaf, subsample in product(
    [100, 200],
    [0.03, 0.05],
    [2, 3],
    [10, 30],
    [0.8],
):
    GB_GRID.append({
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "subsample": subsample,
    })

grid_df = pd.DataFrame(GB_GRID)
print(f"Cantidad de corridas planificadas: {len(grid_df)}")
display(grid_df)

Cantidad de corridas planificadas: 16


,n_estimators,learning_rate,max_depth,min_samples_leaf,subsample
0,100,0.03,2,10,0.8
1,100,0.03,2,30,0.8
2,100,0.03,3,10,0.8
3,100,0.03,3,30,0.8
4,100,0.05,2,10,0.8
5,100,0.05,2,30,0.8
6,100,0.05,3,10,0.8
7,100,0.05,3,30,0.8
8,200,0.03,2,10,0.8
9,200,0.03,2,30,0.8


## Rutas del proyecto

In [4]:
def resolve_project_dir():
    cwd = os.getcwd()
    candidates = [
        cwd,
        os.path.dirname(cwd),
        os.path.join(cwd, "Mburicao_Iberamia"),
        os.path.join(os.path.dirname(cwd), "Mburicao_Iberamia"),
        r"G:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia",
        r"D:\Google Drive\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia",
        "/data/students/federico.moran",
        "/data/students/federico.moran/notebooks",
    ]

    for candidate in candidates:
        if not candidate:
            continue
        if os.path.exists(os.path.join(candidate, "notebooks")) or os.path.exists(os.path.join(candidate, "data")):
            return os.path.abspath(candidate)

    return os.path.abspath(cwd)


PROJECT_DIR = resolve_project_dir()

MODELS_DIR = os.path.join(PROJECT_DIR, "models", "23_gradient_boosting_arx")
REPORTS_DIR = os.path.join(PROJECT_DIR, "reports", "23_gradient_boosting_arx")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)


def find_csv(filename):
    candidates = [
        os.path.join(PROJECT_DIR, "data", "processed", "datos_lluvia_nivel", filename),
        os.path.join(PROJECT_DIR, "datasets", "datos_lluvia_nivel", filename),
        os.path.join(os.path.dirname(PROJECT_DIR), "datasets", "datos_lluvia_nivel", filename),
        os.path.join("/data/students/federico.moran", "datasets", "datos_lluvia_nivel", filename),
        os.path.join("/data/students/federico.moran", "notebooks", "datasets", "datos_lluvia_nivel", filename),
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    raise FileNotFoundError("No se encontro el archivo: " + filename + "\nCandidatos:\n" + "\n".join(candidates))


CSV_PATHS = {
    "2021": find_csv("Mburicao_2021.csv"),
    "2025": find_csv("Mburicao_2025.csv"),
}

print("PROJECT_DIR:", PROJECT_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("CSV encontrados:")
for segment_name, path in CSV_PATHS.items():
    print(segment_name, "->", path)

PROJECT_DIR: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia
MODELS_DIR: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\models\23_gradient_boosting_arx
REPORTS_DIR: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx
CSV encontrados:
2021 -> g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\data\processed\datos_lluvia_nivel\Mburicao_2021.csv
2025 -> g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\data\processed\datos_lluvia_nivel\Mburicao_2025.csv


## Carga y regularizacion de datos

In [5]:
def load_segment(csv_path, segment_name):
    df = pd.read_csv(csv_path)

    required_columns = {TIME_COL, TARGET_COL, RAIN_COL}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Faltan columnas requeridas en {csv_path}: {sorted(missing_columns)}")

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    df = df.sort_values(TIME_COL)
    df = df.set_index(TIME_COL)
    df = df[~df.index.duplicated(keep="first")]

    df = df[[TARGET_COL, RAIN_COL]].copy()
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df[RAIN_COL] = pd.to_numeric(df[RAIN_COL], errors="coerce")

    df = df.asfreq(FREQ)
    df[TARGET_COL] = df[TARGET_COL].interpolate(method="time").ffill().bfill()
    df[RAIN_COL] = df[RAIN_COL].fillna(0.0)
    df["segmento"] = segment_name

    return df


segments = {}
for segment_name, csv_path in CSV_PATHS.items():
    segments[segment_name] = load_segment(csv_path, segment_name)

segment_names = list(segments.keys())

summary_rows = []
for segment_name, df_segment in segments.items():
    summary_rows.append({
        "segmento": segment_name,
        "filas": len(df_segment),
        "inicio": df_segment.index.min(),
        "fin": df_segment.index.max(),
        "freq_inferida": pd.infer_freq(df_segment.index),
        "nivel_min": df_segment[TARGET_COL].min(),
        "nivel_max": df_segment[TARGET_COL].max(),
        "lluvia_total": df_segment[RAIN_COL].sum(),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_path = os.path.join(REPORTS_DIR, "23_dataset_resumen.csv")
summary_df.to_csv(summary_path, index=False)
print("Resumen guardado en:", summary_path)

,segmento,filas,inicio,fin,freq_inferida,nivel_min,nivel_max,lluvia_total
0,2021,42220,2021-06-12 12:00:00,2022-04-01 16:30:00,10min,0.030263,3.746,741.6
1,2025,29055,2025-08-27 15:10:00,2026-03-17 09:30:00,10min,0.400000,3.300,317.2


Resumen guardado en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_dataset_resumen.csv


## Split train-validation-test

In [6]:
split_info = {}

for segment_name in segment_names:
    df_segment = segments[segment_name]
    n = len(df_segment)

    train_end_position = int(n * TRAIN_FRACTION)
    val_end_position = int(n * (TRAIN_FRACTION + VAL_FRACTION))

    min_history = max(9, FORECAST_HORIZON + 1)

    train_end_position = max(train_end_position, min_history + FORECAST_HORIZON + 1)
    val_end_position = max(val_end_position, train_end_position + FORECAST_HORIZON + 1)
    val_end_position = min(val_end_position, n - FORECAST_HORIZON - 1)

    train_end_time = df_segment.index[train_end_position]
    test_start_time = df_segment.index[val_end_position]

    split_info[segment_name] = {
        "train_end_position": train_end_position,
        "val_end_position": val_end_position,
        "test_start_position": val_end_position,
        "train_end_time": train_end_time,
        "test_start_time": test_start_time,
        "train_points": train_end_position,
        "val_points": val_end_position - train_end_position,
        "test_points": n - val_end_position,
    }

split_df = pd.DataFrame([
    {
        "segmento": segment_name,
        "train_end_position": info["train_end_position"],
        "val_end_position": info["val_end_position"],
        "train_end_time": info["train_end_time"],
        "test_start_time": info["test_start_time"],
        "train_points": info["train_points"],
        "val_points": info["val_points"],
        "test_points": info["test_points"],
    }
    for segment_name, info in split_info.items()
])

display(split_df)

split_path = os.path.join(REPORTS_DIR, "23_split_resumen.csv")
split_df.to_csv(split_path, index=False)
print("Split guardado en:", split_path)

,segmento,train_end_position,val_end_position,train_end_time,test_start_time,train_points,val_points,test_points
0,2021,29553,35887,2022-01-03 17:30:00,2022-02-16 17:10:00,29553,6334,6333
1,2025,20338,24696,2026-01-15 20:50:00,2026-02-15 03:10:00,20338,4358,4359


Split guardado en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_split_resumen.csv


## Features ARX

## Construccion de variables predictoras

Las features ARX condensan la historia reciente en variables tabulares. Esto permite que un modelo como Gradient Boosting, que no procesa secuencias directamente como una LSTM, reciba informacion temporal relevante.

Las variables se agrupan en cuatro familias:

### 1. Estado actual del cauce

```text
nivel_t
```

Representa el ultimo nivel observado antes de emitir la prediccion. Es una variable fundamental porque el nivel futuro suele depender fuertemente del nivel actual.

### 2. Cambios recientes del nivel

```text
delta_10m
delta_20m
delta_30m
delta_60m
delta_90m
```

Estas variables capturan si el cauce esta subiendo, bajando o estable. En alerta temprana, un nivel moderado pero con ascenso rapido puede ser mas relevante que un nivel alto pero estable.

### 3. Suavizados locales del nivel

```text
nivel_media_30m
nivel_media_60m
nivel_media_90m
```

Estas medias moviles resumen el comportamiento reciente del cauce y reducen sensibilidad a fluctuaciones puntuales.

### 4. Precipitacion historica

```text
lluvia_10m
lluvia_30m
lluvia_60m
lluvia_90m
lluvia_max_30m
lluvia_max_60m
lluvia_max_90m
```

Estas variables representan la forzante hidrologica principal. Se incluyen sumas y maximos para capturar tanto acumulacion como intensidad reciente.

### 5. Hora del dia

```text
hora_sin
hora_cos
```

La hora se codifica de forma ciclica para evitar discontinuidades artificiales entre 23:50 y 00:00. Esta codificacion permite que el modelo represente patrones horarios si existen.

Todas estas features se calculan dentro de cada segmento. No hay ninguna feature que mezcle informacion entre 2021 y 2025.

In [7]:
def safe_slice(values, start, end):
    start = max(0, start)
    end = max(start, end)
    return values[start:end]


def mean_or_last(values, fallback):
    if len(values) == 0:
        return fallback
    return float(np.mean(values))


def sum_or_zero(values):
    if len(values) == 0:
        return 0.0
    return float(np.sum(values))


def max_or_zero(values):
    if len(values) == 0:
        return 0.0
    return float(np.max(values))


def make_arx_features(df_segment, origin):
    y = df_segment[TARGET_COL].values.astype(float)
    rain = df_segment[RAIN_COL].values.astype(float)
    timestamp = df_segment.index[origin - 1]

    last_level = y[origin - 1]

    def lag_delta(steps):
        if origin > steps:
            return last_level - y[origin - 1 - steps]
        return 0.0

    level_30m = safe_slice(y, origin - 3, origin)
    level_60m = safe_slice(y, origin - 6, origin)
    level_90m = safe_slice(y, origin - 9, origin)

    rain_10m = safe_slice(rain, origin - 1, origin)
    rain_30m = safe_slice(rain, origin - 3, origin)
    rain_60m = safe_slice(rain, origin - 6, origin)
    rain_90m = safe_slice(rain, origin - 9, origin)

    hour = timestamp.hour + timestamp.minute / 60
    hour_angle = 2 * np.pi * hour / 24

    return np.array([
        last_level,
        lag_delta(1),
        lag_delta(2),
        lag_delta(3),
        lag_delta(6),
        lag_delta(9),
        mean_or_last(level_30m, last_level),
        mean_or_last(level_60m, last_level),
        mean_or_last(level_90m, last_level),
        sum_or_zero(rain_10m),
        sum_or_zero(rain_30m),
        sum_or_zero(rain_60m),
        sum_or_zero(rain_90m),
        max_or_zero(rain_30m),
        max_or_zero(rain_60m),
        max_or_zero(rain_90m),
        np.sin(hour_angle),
        np.cos(hour_angle),
    ], dtype=float)


FEATURE_NAMES = [
    "nivel_t",
    "delta_10m",
    "delta_20m",
    "delta_30m",
    "delta_60m",
    "delta_90m",
    "nivel_media_30m",
    "nivel_media_60m",
    "nivel_media_90m",
    "lluvia_10m",
    "lluvia_30m",
    "lluvia_60m",
    "lluvia_90m",
    "lluvia_max_30m",
    "lluvia_max_60m",
    "lluvia_max_90m",
    "hora_sin",
    "hora_cos",
]

display(pd.DataFrame({"feature": FEATURE_NAMES}))

,feature
0,nivel_t
1,delta_10m
2,delta_20m
3,delta_30m
4,delta_60m
5,delta_90m
6,nivel_media_30m
7,nivel_media_60m
8,nivel_media_90m
9,lluvia_10m


## Datasets supervisados

## De serie temporal a tabla supervisada

El modelo no recibe la serie completa como una secuencia. Recibe una tabla donde cada fila corresponde a un origen de prediccion.

Para cada origen `t`, se construye:

```text
X_i = features historicas calculadas hasta t
y_i = [Nivel(t+10), Nivel(t+20), Nivel(t+30)]
```

Esto produce cuatro matrices principales:

```text
X_train, y_train
X_val, y_val
X_test, y_test
```

La matriz `X_train` contiene ejemplos de train de 2021 y de 2025. El modelo final es global porque aprende de ambos segmentos:

```text
X_train = features_train_2021 + features_train_2025
y_train = futuros_train_2021 + futuros_train_2025
```

Pero cada ejemplo se construye internamente dentro de su segmento. Esto significa que no hay continuidad artificial entre anos.

El conjunto `validation` se usa exclusivamente para seleccionar hiperparametros. El conjunto `test` queda reservado para reportar las metricas finales.

Por defecto, `FINAL_TRAIN_USES_VALIDATION = False`, para mantener una separacion estricta: el modelo final se entrena con train y se evalua en test, usando validation solo para seleccion. Si se quisiera maximizar datos de entrenamiento despues de seleccionar hiperparametros, podria activarse `True`, pero eso cambiaria la convencion respecto a los experimentos finales anteriores.

In [8]:
def build_supervised_dataset(split_name):
    X_rows = []
    y_rows = []
    meta_rows = []
    min_history = 9

    for segment_name in segment_names:
        df_segment = segments[segment_name]
        info = split_info[segment_name]
        values = df_segment[TARGET_COL].values.astype(float)
        time_index = df_segment.index

        if split_name == "train":
            start_origin = min_history
            end_origin = info["train_end_position"] - FORECAST_HORIZON
        elif split_name == "validation":
            start_origin = info["train_end_position"]
            end_origin = info["test_start_position"] - FORECAST_HORIZON
        elif split_name == "test":
            start_origin = info["test_start_position"]
            end_origin = len(df_segment) - FORECAST_HORIZON
        elif split_name == "train_validation":
            start_origin = min_history
            end_origin = info["test_start_position"] - FORECAST_HORIZON
        else:
            raise ValueError("split_name no reconocido: " + split_name)

        start_origin = max(start_origin, min_history)

        for origin in range(start_origin, end_origin + 1, EVAL_STRIDE):
            X_rows.append(make_arx_features(df_segment, origin))
            y_rows.append(values[origin:origin + FORECAST_HORIZON])
            meta_rows.append({
                "segmento": segment_name,
                "split": split_name,
                "origin": origin,
                "origin_time": time_index[origin - 1],
            })

    return np.vstack(X_rows), np.vstack(y_rows), pd.DataFrame(meta_rows)


X_train, y_train, meta_train = build_supervised_dataset("train")
X_val, y_val, meta_val = build_supervised_dataset("validation")
X_test, y_test, meta_test = build_supervised_dataset("test")

if FINAL_TRAIN_USES_VALIDATION:
    X_final_train, y_final_train, meta_final_train = build_supervised_dataset("train_validation")
else:
    X_final_train, y_final_train, meta_final_train = X_train, y_train, meta_train

dataset_summary_df = pd.DataFrame([
    {"split": "train", "n_windows": len(X_train), "X_shape": str(tuple(X_train.shape)), "y_shape": str(tuple(y_train.shape))},
    {"split": "validation", "n_windows": len(X_val), "X_shape": str(tuple(X_val.shape)), "y_shape": str(tuple(y_val.shape))},
    {"split": "test", "n_windows": len(X_test), "X_shape": str(tuple(X_test.shape)), "y_shape": str(tuple(y_test.shape))},
    {"split": "final_train", "n_windows": len(X_final_train), "X_shape": str(tuple(X_final_train.shape)), "y_shape": str(tuple(y_final_train.shape))},
])

display(dataset_summary_df)

dataset_summary_path = os.path.join(REPORTS_DIR, "23_dataset_supervisado_resumen.csv")
dataset_summary_df.to_csv(dataset_summary_path, index=False)
print("Resumen de dataset supervisado guardado en:", dataset_summary_path)

,split,n_windows,X_shape,y_shape
0,train,49869,"(49869, 18)","(49869, 3)"
1,validation,10688,"(10688, 18)","(10688, 3)"
2,test,10688,"(10688, 18)","(10688, 3)"
3,final_train,49869,"(49869, 18)","(49869, 3)"


Resumen de dataset supervisado guardado en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_dataset_supervisado_resumen.csv


## Utilidades de prediccion y metricas

## Formato largo de predicciones

Aunque el modelo produce tres valores por ejemplo, las predicciones se guardan en formato largo para mantener compatibilidad con los otros notebooks.

Una fila supervisada produce tres filas de prediccion:

```text
lead_minutes = 10
lead_minutes = 20
lead_minutes = 30
```

Cada fila contiene:

- `segmento`: 2021 o 2025;
- `modelo`: `GradientBoosting_ARX`;
- `origin_time`: ultimo instante observado antes de predecir;
- `pred_time`: instante futuro predicho;
- `lead_step`: horizonte en pasos;
- `lead_minutes`: horizonte en minutos;
- `pred`: nivel predicho;
- `actual`: nivel observado;
- `Precipitacion`: lluvia observada en `pred_time`, usada para analisis posterior;
- `lluvia_1h`: lluvia acumulada reciente;
- `delta_nivel`: cambio observado del nivel.

Las columnas de contexto no se usan para entrenar el modelo en test. Se agregan despues para calcular metricas por condiciones criticas.

In [9]:
EPS = 1e-8


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "MAPE (%)": np.nan,
            "sMAPE (%)": np.nan,
            "Bias": np.nan,
            "NSE": np.nan,
        }

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan

    valid_mape = np.abs(y_true) > EPS
    mape = np.mean(np.abs((y_true[valid_mape] - y_pred[valid_mape]) / y_true[valid_mape])) * 100 if valid_mape.any() else np.nan

    smape_denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    valid_smape = smape_denom > EPS
    smape = np.mean(np.abs(y_true[valid_smape] - y_pred[valid_smape]) / smape_denom[valid_smape]) * 100 if valid_smape.any() else np.nan

    bias = np.mean(y_pred - y_true)
    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    nse = 1 - np.sum((y_true - y_pred) ** 2) / denominator if denominator > EPS else np.nan

    return {
        "n": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE (%)": mape,
        "sMAPE (%)": smape,
        "Bias": bias,
        "NSE": nse,
    }


def context_for_segment(segment_name):
    df_segment = segments[segment_name]
    context = df_segment[[TARGET_COL, RAIN_COL]].copy()
    context["lluvia_1h"] = context[RAIN_COL].rolling(STEPS_PER_HOUR, min_periods=1).sum()
    context["delta_nivel"] = context[TARGET_COL].diff()
    context = context.rename(columns={TARGET_COL: "actual"})
    context["pred_time"] = context.index
    return context


def predictions_to_long(meta_df, pred_matrix, model_name):
    rows = []

    for row_position, meta in meta_df.reset_index(drop=True).iterrows():
        segment_name = meta["segmento"]
        origin = int(meta["origin"])
        df_segment = segments[segment_name]
        time_index = df_segment.index
        origin_time = meta["origin_time"]
        forecast = pred_matrix[row_position]

        for lead_step in range(1, FORECAST_HORIZON + 1):
            pred_index = origin + lead_step - 1
            rows.append({
                "segmento": segment_name,
                "modelo": model_name,
                "origin_time": origin_time,
                "pred_time": time_index[pred_index],
                "lead_step": lead_step,
                "lead_minutes": lead_step * 10,
                "pred": float(forecast[lead_step - 1]),
            })

    pred_df = pd.DataFrame(rows)
    if pred_df.empty:
        return pred_df

    enriched_parts = []
    for segment_name, segment_pred in pred_df.groupby("segmento"):
        context = context_for_segment(segment_name)
        enriched = segment_pred.merge(
            context[["pred_time", "actual", RAIN_COL, "lluvia_1h", "delta_nivel"]],
            on="pred_time",
            how="left",
        )
        enriched_parts.append(enriched)

    return pd.concat(enriched_parts, axis=0).dropna(subset=["actual", "pred"]).reset_index(drop=True)


def build_metrics_table_from_predictions(pred_df):
    rows = []
    global_parts = []

    for segment_name in segment_names:
        eval_df = pred_df[pred_df["segmento"] == segment_name]
        if eval_df.empty:
            continue
        metrics = compute_metrics(eval_df["actual"].values, eval_df["pred"].values)
        metrics["modelo"] = MODEL_NAME
        metrics["segmento"] = segment_name
        rows.append(metrics)
        global_parts.append(eval_df)

    if global_parts:
        global_df = pd.concat(global_parts, axis=0)
        metrics = compute_metrics(global_df["actual"].values, global_df["pred"].values)
        metrics["modelo"] = MODEL_NAME
        metrics["segmento"] = "global"
        rows.append(metrics)

    cols = ["modelo", "segmento", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    return pd.DataFrame(rows)[cols].sort_values(["segmento", "RMSE", "modelo"]).reset_index(drop=True)


def build_lead_metrics_table_from_predictions(pred_df):
    rows = []
    global_parts = []

    for segment_name in segment_names:
        eval_df = pred_df[pred_df["segmento"] == segment_name]
        if eval_df.empty:
            continue

        for lead_minutes, lead_df in eval_df.groupby("lead_minutes"):
            metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
            metrics["modelo"] = MODEL_NAME
            metrics["segmento"] = segment_name
            metrics["lead_minutes"] = int(lead_minutes)
            rows.append(metrics)

        global_parts.append(eval_df)

    if global_parts:
        global_df = pd.concat(global_parts, axis=0)
        for lead_minutes, lead_df in global_df.groupby("lead_minutes"):
            metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
            metrics["modelo"] = MODEL_NAME
            metrics["segmento"] = "global"
            metrics["lead_minutes"] = int(lead_minutes)
            rows.append(metrics)

    cols = ["modelo", "segmento", "lead_minutes", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    return pd.DataFrame(rows)[cols].sort_values(["segmento", "lead_minutes", "RMSE", "modelo"]).reset_index(drop=True)

## Entrenamiento con grid search

## Que ocurre en cada corrida del grid search

Cada configuracion del grid define un ensamble Gradient Boosting. Como la salida tiene tres horizontes, `MultiOutputRegressor` ajusta internamente tres regresores:

```text
regresor_10min.fit(X_train, y_train[:, 0])
regresor_20min.fit(X_train, y_train[:, 1])
regresor_30min.fit(X_train, y_train[:, 2])
```

Durante una corrida:

1. Se crea el modelo con una combinacion de hiperparametros.
2. Se entrena con `X_train` e `y_train`.
3. Se predice sobre `X_val`.
4. Las predicciones de validation se convierten a formato largo.
5. Se calculan metricas globales y por segmento.
6. Se guardan las metricas junto con el `run_id`.

La mejor configuracion se elige por menor RMSE global en validation. Este criterio es consistente con los otros notebooks, donde validation se usa para seleccion y test para evaluacion final.

In [10]:
def build_model(params):
    base_model = GradientBoostingRegressor(
        n_estimators=int(params["n_estimators"]),
        learning_rate=float(params["learning_rate"]),
        max_depth=int(params["max_depth"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        subsample=float(params["subsample"]),
        random_state=RANDOM_STATE,
    )
    return MultiOutputRegressor(base_model, n_jobs=-1)


grid_metrics_rows = []
grid_run_rows = []

for run_index, params in enumerate(GB_GRID, start=1):
    run_id = (
        f"run_{run_index:03d}"
        f"_ne{params['n_estimators']}"
        f"_lr{str(params['learning_rate']).replace('.', 'p')}"
        f"_md{params['max_depth']}"
        f"_leaf{params['min_samples_leaf']}"
        f"_sub{str(params['subsample']).replace('.', 'p')}"
    )

    print("=" * 100)
    print("Iniciando:", run_id)
    print(params)

    start_time = time.time()
    status = "ok"
    error_message = ""

    try:
        model_candidate = build_model(params)
        model_candidate.fit(X_train, y_train)

        val_pred_matrix = model_candidate.predict(X_val)
        val_pred_long = predictions_to_long(meta_val, val_pred_matrix, MODEL_NAME)
        val_metrics = build_metrics_table_from_predictions(val_pred_long)
        val_metrics["run_id"] = run_id

        for key, value in params.items():
            val_metrics[key] = value

        grid_metrics_rows.append(val_metrics)
        global_metrics = val_metrics[val_metrics["segmento"] == "global"].iloc[0]
        validation_rmse_global = float(global_metrics["RMSE"])
        validation_mae_global = float(global_metrics["MAE"])
        validation_nse_global = float(global_metrics["NSE"])

    except Exception as exc:
        status = "error"
        error_message = str(exc)
        validation_rmse_global = np.nan
        validation_mae_global = np.nan
        validation_nse_global = np.nan
        print("ERROR:", error_message)

    elapsed_seconds = time.time() - start_time

    grid_run_rows.append({
        "run_id": run_id,
        "status": status,
        "error_message": error_message,
        "elapsed_seconds": elapsed_seconds,
        "validation_RMSE_global": validation_rmse_global,
        "validation_MAE_global": validation_mae_global,
        "validation_NSE_global": validation_nse_global,
        **params,
    })

    print("Finalizado:", run_id, "| status:", status, "| segundos:", round(elapsed_seconds, 1))

grid_metrics_df = pd.concat(grid_metrics_rows, axis=0, ignore_index=True) if grid_metrics_rows else pd.DataFrame()
grid_runs_df = pd.DataFrame(grid_run_rows)

grid_metrics_path = os.path.join(REPORTS_DIR, "23_gradient_boosting_arx_gridsearch_metricas.csv")
grid_runs_path = os.path.join(REPORTS_DIR, "23_gradient_boosting_arx_gridsearch_resumen_corridas.csv")

grid_metrics_df.to_csv(grid_metrics_path, index=False)
grid_runs_df.to_csv(grid_runs_path, index=False)

print("Metricas de grid search guardadas en:", grid_metrics_path)
print("Resumen de corridas guardado en:", grid_runs_path)

display(grid_runs_df.sort_values("validation_RMSE_global").head(10))

Iniciando: run_001_ne100_lr0p03_md2_leaf10_sub0p8
{'n_estimators': 100, 'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 10, 'subsample': 0.8}
Finalizado: run_001_ne100_lr0p03_md2_leaf10_sub0p8 | status: ok | segundos: 41.3
Iniciando: run_002_ne100_lr0p03_md2_leaf30_sub0p8
{'n_estimators': 100, 'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 30, 'subsample': 0.8}
Finalizado: run_002_ne100_lr0p03_md2_leaf30_sub0p8 | status: ok | segundos: 21.4
Iniciando: run_003_ne100_lr0p03_md3_leaf10_sub0p8
{'n_estimators': 100, 'learning_rate': 0.03, 'max_depth': 3, 'min_samples_leaf': 10, 'subsample': 0.8}
Finalizado: run_003_ne100_lr0p03_md3_leaf10_sub0p8 | status: ok | segundos: 12.9
Iniciando: run_004_ne100_lr0p03_md3_leaf30_sub0p8
{'n_estimators': 100, 'learning_rate': 0.03, 'max_depth': 3, 'min_samples_leaf': 30, 'subsample': 0.8}
Finalizado: run_004_ne100_lr0p03_md3_leaf30_sub0p8 | status: ok | segundos: 12.8
Iniciando: run_005_ne100_lr0p05_md2_leaf10_sub0p8
{'n_estimators

,run_id,status,error_message,elapsed_seconds,validation_RMSE_global,validation_MAE_global,validation_NSE_global,n_estimators,learning_rate,max_depth,min_samples_leaf,subsample
13,run_014_ne200_lr0p05_md2_leaf30_sub0p8,ok,,17.066288,0.042040,0.005078,0.969418,200,0.05,2,30,0.8
11,run_012_ne200_lr0p03_md3_leaf30_sub0p8,ok,,25.283291,0.042721,0.005216,0.968419,200,0.03,3,30,0.8
9,run_010_ne200_lr0p03_md2_leaf30_sub0p8,ok,,17.125738,0.043360,0.006076,0.967467,200,0.03,2,30,0.8
15,run_016_ne200_lr0p05_md3_leaf30_sub0p8,ok,,25.006903,0.043379,0.004779,0.967439,200,0.05,3,30,0.8
7,run_008_ne100_lr0p05_md3_leaf30_sub0p8,ok,,11.932221,0.043603,0.005745,0.967101,100,0.05,3,30,0.8
14,run_015_ne200_lr0p05_md3_leaf10_sub0p8,ok,,24.132614,0.044108,0.004782,0.966335,200,0.05,3,10,0.8
12,run_013_ne200_lr0p05_md2_leaf10_sub0p8,ok,,16.939698,0.044203,0.005184,0.966190,200,0.05,2,10,0.8
10,run_011_ne200_lr0p03_md3_leaf10_sub0p8,ok,,25.839399,0.044285,0.005343,0.966065,200,0.03,3,10,0.8
6,run_007_ne100_lr0p05_md3_leaf10_sub0p8,ok,,13.239546,0.044940,0.005845,0.965053,100,0.05,3,10,0.8
5,run_006_ne100_lr0p05_md2_leaf30_sub0p8,ok,,9.055724,0.044951,0.006905,0.965036,100,0.05,2,30,0.8


## Seleccion de mejor configuracion

## Criterio de seleccion

La seleccion se realiza sobre las filas de validation con `segmento = global`. La metrica principal es RMSE.

La metrica global se calcula uniendo las predicciones de validation de 2021 y 2025, no promediando los RMSE por separado. Esto mantiene el mismo criterio usado en los demas modelos:

```text
predicciones_validation_global =
    predicciones_validation_2021 + predicciones_validation_2025

RMSE_global = RMSE(actual_global, pred_global)
```

La configuracion seleccionada se guarda en `23_gradient_boosting_arx_gridsearch_mejor_config.csv`, junto con sus metricas de validation. Esta tabla permite documentar que el test no fue usado para elegir hiperparametros.

In [11]:
if grid_metrics_df.empty:
    raise RuntimeError("El grid search no produjo metricas validas.")

global_grid = grid_metrics_df[grid_metrics_df["segmento"] == "global"].copy()
global_grid = global_grid.sort_values(["RMSE", "MAE", "run_id"]).reset_index(drop=True)
best_row = global_grid.iloc[0]

best_params = {
    "n_estimators": int(best_row["n_estimators"]),
    "learning_rate": float(best_row["learning_rate"]),
    "max_depth": int(best_row["max_depth"]),
    "min_samples_leaf": int(best_row["min_samples_leaf"]),
    "subsample": float(best_row["subsample"]),
}

best_config_df = pd.DataFrame([{
    "selected_run_id": best_row["run_id"],
    **best_params,
    "validation_n": int(best_row["n"]),
    "validation_MAE": float(best_row["MAE"]),
    "validation_RMSE": float(best_row["RMSE"]),
    "validation_R2": float(best_row["R2"]),
    "validation_NSE": float(best_row["NSE"]),
    "validation_Bias": float(best_row["Bias"]),
}])

best_config_path = os.path.join(REPORTS_DIR, "23_gradient_boosting_arx_gridsearch_mejor_config.csv")
best_config_df.to_csv(best_config_path, index=False)

display(global_grid.head(20))
display(best_config_df)
print("Mejor configuracion guardada en:", best_config_path)

,modelo,segmento,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE,run_id,n_estimators,learning_rate,max_depth,min_samples_leaf,subsample
0,GradientBoosting_ARX,global,32064,0.005078,0.042040,0.969418,1.620174,1.633700,-0.002041,0.969418,run_014_ne200_lr0p05_md2_leaf30_sub0p8,200,0.05,2,30,0.8
1,GradientBoosting_ARX,global,32064,0.005216,0.042721,0.968419,1.785583,1.798891,-0.002074,0.968419,run_012_ne200_lr0p03_md3_leaf30_sub0p8,200,0.03,3,30,0.8
2,GradientBoosting_ARX,global,32064,0.006076,0.043360,0.967467,2.287234,2.285418,-0.002481,0.967467,run_010_ne200_lr0p03_md2_leaf30_sub0p8,200,0.03,2,30,0.8
3,GradientBoosting_ARX,global,32064,0.004779,0.043379,0.967439,1.387824,1.409326,-0.002068,0.967439,run_016_ne200_lr0p05_md3_leaf30_sub0p8,200,0.05,3,30,0.8
4,GradientBoosting_ARX,global,32064,0.005745,0.043603,0.967101,2.158437,2.165357,-0.002216,0.967101,run_008_ne100_lr0p05_md3_leaf30_sub0p8,100,0.05,3,30,0.8
5,GradientBoosting_ARX,global,32064,0.004782,0.044108,0.966335,1.345047,1.363372,-0.001936,0.966335,run_015_ne200_lr0p05_md3_leaf10_sub0p8,200,0.05,3,10,0.8
6,GradientBoosting_ARX,global,32064,0.005184,0.044203,0.966190,1.652273,1.666519,-0.002030,0.966190,run_013_ne200_lr0p05_md2_leaf10_sub0p8,200,0.05,2,10,0.8
7,GradientBoosting_ARX,global,32064,0.005343,0.044285,0.966065,1.799225,1.810652,-0.002036,0.966065,run_011_ne200_lr0p03_md3_leaf10_sub0p8,200,0.03,3,10,0.8
8,GradientBoosting_ARX,global,32064,0.005845,0.044940,0.965053,2.150483,2.156079,-0.002244,0.965053,run_007_ne100_lr0p05_md3_leaf10_sub0p8,100,0.05,3,10,0.8
9,GradientBoosting_ARX,global,32064,0.006905,0.044951,0.965036,2.788903,2.772505,-0.002916,0.965036,run_006_ne100_lr0p05_md2_leaf30_sub0p8,100,0.05,2,30,0.8


,selected_run_id,n_estimators,learning_rate,max_depth,min_samples_leaf,subsample,validation_n,validation_MAE,validation_RMSE,validation_R2,validation_NSE,validation_Bias
0,run_014_ne200_lr0p05_md2_leaf30_sub0p8,200,0.05,2,30,0.8,32064,0.005078,0.04204,0.969418,0.969418,-0.002041


Mejor configuracion guardada en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_gradient_boosting_arx_gridsearch_mejor_config.csv


## Entrenamiento final y predicciones de test

## Entrenamiento final

Una vez seleccionados los hiperparametros, se entrena el modelo final con la mejor configuracion. Este modelo es el que se usa para generar las predicciones sobre test.

El objeto guardado incluye:

- el modelo entrenado;
- los mejores hiperparametros;
- los nombres de features;
- el horizonte de prediccion;
- la informacion de split.

Las predicciones de test se guardan en formato largo para que luego puedan combinarse con reportes de Ridge-ARX, N-BEATS, ARIMA, SARIMAX y LSTM.

Este notebook no calcula comparaciones contra otros modelos. Solamente deja las salidas de Gradient Boosting-ARX listas y compatibles.

In [12]:
final_model = build_model(best_params)
final_model.fit(X_final_train, y_final_train)

test_pred_matrix = final_model.predict(X_test)
test_predictions_long_df = predictions_to_long(meta_test, test_pred_matrix, MODEL_NAME)

predictions_path = os.path.join(REPORTS_DIR, "23_predicciones_test_long.csv")
test_predictions_long_df.to_csv(predictions_path, index=False)

model_path = os.path.join(MODELS_DIR, "23_gradient_boosting_arx.joblib")
joblib.dump(
    {
        "model": final_model,
        "best_params": best_params,
        "feature_names": FEATURE_NAMES,
        "forecast_horizon": FORECAST_HORIZON,
        "final_train_uses_validation": FINAL_TRAIN_USES_VALIDATION,
        "split_info": split_info,
    },
    model_path,
)

display(test_predictions_long_df.head())
print("Predicciones de test guardadas en:", predictions_path)
print("Modelo guardado en:", model_path)

,segmento,modelo,origin_time,pred_time,lead_step,lead_minutes,pred,actual,Precipitacion,lluvia_1h,delta_nivel
0,2021,GradientBoosting_ARX,2022-02-16 17:00:00,2022-02-16 17:10:00,1,10,0.091399,0.092288,0.0,0.0,0.000615
1,2021,GradientBoosting_ARX,2022-02-16 17:00:00,2022-02-16 17:20:00,2,20,0.090416,0.090951,0.0,0.0,-0.001337
2,2021,GradientBoosting_ARX,2022-02-16 17:00:00,2022-02-16 17:30:00,3,30,0.090202,0.089209,0.0,0.0,-0.001742
3,2021,GradientBoosting_ARX,2022-02-16 17:10:00,2022-02-16 17:20:00,1,10,0.091399,0.090951,0.0,0.0,-0.001337
4,2021,GradientBoosting_ARX,2022-02-16 17:10:00,2022-02-16 17:30:00,2,20,0.090416,0.089209,0.0,0.0,-0.001742


Predicciones de test guardadas en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_predicciones_test_long.csv
Modelo guardado en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\models\23_gradient_boosting_arx\23_gradient_boosting_arx.joblib


## Metricas de test

## Metricas finales de test

Las metricas finales se calculan sobre test, usando unidades originales del nivel. Se generan dos tablas principales:

1. `23_metricas_global_segmento.csv`: metricas agregadas por segmento y globales.
2. `23_metricas_por_lead.csv`: metricas separadas para t+10, t+20 y t+30 minutos.

Las metricas reportadas son:

- `MAE`: error absoluto medio;
- `RMSE`: raiz del error cuadratico medio;
- `R2`: proporcion de varianza explicada;
- `MAPE (%)`: error porcentual absoluto medio;
- `sMAPE (%)`: error porcentual simetrico;
- `Bias`: promedio de `pred - actual`;
- `NSE`: eficiencia de Nash-Sutcliffe.

La tabla global permite resumir desempeno general. La tabla por horizonte permite estudiar si el error aumenta de forma progresiva hacia t+30 minutos.

In [13]:
metrics_df = build_metrics_table_from_predictions(test_predictions_long_df)
lead_metrics_df = build_lead_metrics_table_from_predictions(test_predictions_long_df)

metrics_path = os.path.join(REPORTS_DIR, "23_metricas_global_segmento.csv")
lead_metrics_path = os.path.join(REPORTS_DIR, "23_metricas_por_lead.csv")

metrics_df.to_csv(metrics_path, index=False)
lead_metrics_df.to_csv(lead_metrics_path, index=False)

display(metrics_df)
display(lead_metrics_df)

print("Metricas globales/segmento guardadas en:", metrics_path)
print("Metricas por lead guardadas en:", lead_metrics_path)

,modelo,segmento,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE
0,GradientBoosting_ARX,2021,18993,0.013060,0.082957,0.854783,4.408994,4.408721,-0.002391,0.854783
1,GradientBoosting_ARX,2025,13071,0.019259,0.082860,0.851823,2.266286,2.365964,-0.017226,0.851823
2,GradientBoosting_ARX,global,32064,0.015587,0.082917,0.924152,3.535511,3.575984,-0.008438,0.924152


,modelo,segmento,lead_minutes,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE
0,GradientBoosting_ARX,2021,10,6331,0.009554,0.069492,0.898097,3.150232,3.157577,-0.000846,0.898097
1,GradientBoosting_ARX,2021,20,6331,0.013120,0.081925,0.858372,4.424471,4.422071,-0.002310,0.858372
2,GradientBoosting_ARX,2021,30,6331,0.016508,0.095418,0.807878,5.652278,5.646515,-0.004016,0.807878
3,GradientBoosting_ARX,2025,10,4357,0.013485,0.060727,0.920412,1.661942,1.709702,-0.011479,0.920412
4,GradientBoosting_ARX,2025,20,4357,0.020069,0.082048,0.854713,2.384498,2.476655,-0.017870,0.854713
5,GradientBoosting_ARX,2025,30,4357,0.024224,0.100885,0.780342,2.752417,2.911535,-0.022329,0.780342
6,GradientBoosting_ARX,global,10,10688,0.011156,0.066060,0.951857,2.543525,2.567346,-0.005181,0.951857
7,GradientBoosting_ARX,global,20,10688,0.015953,0.081975,0.925865,3.592869,3.629015,-0.008653,0.925865
8,GradientBoosting_ARX,global,30,10688,0.019653,0.097683,0.894733,4.470140,4.531591,-0.011481,0.894733


Metricas globales/segmento guardadas en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_metricas_global_segmento.csv
Metricas por lead guardadas en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_metricas_por_lead.csv


## Metricas por condiciones criticas

## Evaluacion en condiciones hidrologicamente relevantes

El desempeno promedio puede ocultar fallas durante eventos de interes. Por eso se calculan metricas en condiciones criticas:

- `precipitacion_gt_0`: instantes con lluvia;
- `lluvia_1h_alta`: lluvia acumulada reciente alta;
- `nivel_p90`: niveles altos;
- `delta_positivo_alto`: ascensos rapidos del nivel.

Estas condiciones permiten analizar si Gradient Boosting-ARX captura bien situaciones no lineales y potencialmente peligrosas. Por ejemplo, un buen desempeno global pero mal RMSE en `delta_positivo_alto` indicaria que el modelo funciona en periodos tranquilos, pero falla en ascensos bruscos.

Las condiciones se calculan despues de generar las predicciones. No forman parte del entrenamiento; son una herramienta de evaluacion.

In [14]:
def condition_masks(eval_df):
    masks = {}
    masks["todos"] = pd.Series(True, index=eval_df.index)
    masks["precipitacion_gt_0"] = eval_df[RAIN_COL] > 0

    positive_rain_1h = eval_df.loc[eval_df["lluvia_1h"] > 0, "lluvia_1h"]
    if len(positive_rain_1h) > 0:
        rain_1h_threshold = positive_rain_1h.quantile(0.75)
        masks["lluvia_1h_alta"] = eval_df["lluvia_1h"] >= rain_1h_threshold
    else:
        masks["lluvia_1h_alta"] = pd.Series(False, index=eval_df.index)

    level_threshold = eval_df["actual"].quantile(0.90)
    masks["nivel_p90"] = eval_df["actual"] >= level_threshold

    positive_delta = eval_df.loc[eval_df["delta_nivel"] > 0, "delta_nivel"]
    if len(positive_delta) > 0:
        delta_threshold = positive_delta.quantile(0.90)
        masks["delta_positivo_alto"] = eval_df["delta_nivel"] >= delta_threshold
    else:
        masks["delta_positivo_alto"] = pd.Series(False, index=eval_df.index)

    return masks


def build_condition_metrics_tables(pred_df):
    condition_rows = []
    condition_lead_rows = []
    global_parts = []

    for segment_name in segment_names:
        eval_df = pred_df[pred_df["segmento"] == segment_name]
        if eval_df.empty:
            continue

        global_parts.append(eval_df)
        masks = condition_masks(eval_df)

        for condition_name, mask in masks.items():
            subset = eval_df.loc[mask]
            metrics = compute_metrics(subset["actual"].values, subset["pred"].values)
            metrics["modelo"] = MODEL_NAME
            metrics["segmento"] = segment_name
            metrics["condicion"] = condition_name
            condition_rows.append(metrics)

            for lead_minutes, lead_df in subset.groupby("lead_minutes"):
                lead_metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
                lead_metrics["modelo"] = MODEL_NAME
                lead_metrics["segmento"] = segment_name
                lead_metrics["condicion"] = condition_name
                lead_metrics["lead_minutes"] = int(lead_minutes)
                condition_lead_rows.append(lead_metrics)

    if global_parts:
        global_df = pd.concat(global_parts, axis=0).reset_index(drop=True)
        masks = condition_masks(global_df)

        for condition_name, mask in masks.items():
            subset = global_df.loc[mask]
            metrics = compute_metrics(subset["actual"].values, subset["pred"].values)
            metrics["modelo"] = MODEL_NAME
            metrics["segmento"] = "global"
            metrics["condicion"] = condition_name
            condition_rows.append(metrics)

            for lead_minutes, lead_df in subset.groupby("lead_minutes"):
                lead_metrics = compute_metrics(lead_df["actual"].values, lead_df["pred"].values)
                lead_metrics["modelo"] = MODEL_NAME
                lead_metrics["segmento"] = "global"
                lead_metrics["condicion"] = condition_name
                lead_metrics["lead_minutes"] = int(lead_minutes)
                condition_lead_rows.append(lead_metrics)

    condition_metrics_df = pd.DataFrame(condition_rows)
    condition_metrics_df = condition_metrics_df[
        ["modelo", "segmento", "condicion", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    ].sort_values(["condicion", "segmento", "RMSE", "modelo"]).reset_index(drop=True)

    condition_lead_metrics_df = pd.DataFrame(condition_lead_rows)
    condition_lead_metrics_df = condition_lead_metrics_df[
        ["modelo", "segmento", "condicion", "lead_minutes", "n", "MAE", "RMSE", "R2", "MAPE (%)", "sMAPE (%)", "Bias", "NSE"]
    ].sort_values(["condicion", "segmento", "lead_minutes", "RMSE", "modelo"]).reset_index(drop=True)

    return condition_metrics_df, condition_lead_metrics_df


condition_metrics_df, condition_lead_metrics_df = build_condition_metrics_tables(test_predictions_long_df)

condition_metrics_path = os.path.join(REPORTS_DIR, "23_metricas_condiciones.csv")
condition_lead_metrics_path = os.path.join(REPORTS_DIR, "23_metricas_condiciones_por_lead.csv")

condition_metrics_df.to_csv(condition_metrics_path, index=False)
condition_lead_metrics_df.to_csv(condition_lead_metrics_path, index=False)

display(condition_metrics_df)
display(condition_lead_metrics_df)

print("Metricas por condicion guardadas en:", condition_metrics_path)
print("Metricas por condicion y lead guardadas en:", condition_lead_metrics_path)

,modelo,segmento,condicion,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE
0,GradientBoosting_ARX,2021,delta_positivo_alto,660,0.134034,0.356532,0.716388,17.678750,19.973054,-0.087581,0.716388
1,GradientBoosting_ARX,2025,delta_positivo_alto,72,0.677684,0.781631,-0.573628,34.527964,44.105080,-0.672336,-0.573628
2,GradientBoosting_ARX,global,delta_positivo_alto,795,0.201519,0.422155,0.669692,17.917033,20.750966,-0.156348,0.669692
3,GradientBoosting_ARX,2021,lluvia_1h_alta,267,0.391043,0.661242,0.610195,30.868181,31.856598,-0.245944,0.610195
4,GradientBoosting_ARX,2025,lluvia_1h_alta,159,0.473142,0.630710,0.484203,25.428218,28.943013,-0.403334,0.484203
5,GradientBoosting_ARX,global,lluvia_1h_alta,423,0.427612,0.653658,0.598352,28.993183,30.956190,-0.310806,0.598352
6,GradientBoosting_ARX,2021,nivel_p90,1902,0.086547,0.259669,0.739738,9.636822,10.235532,-0.030251,0.739738
7,GradientBoosting_ARX,2025,nivel_p90,2169,0.070137,0.201169,0.820613,5.197588,5.763632,-0.061270,0.820613
8,GradientBoosting_ARX,global,nivel_p90,4206,0.069585,0.224358,0.766612,5.339302,5.890047,-0.051448,0.766612
9,GradientBoosting_ARX,2021,precipitacion_gt_0,564,0.198464,0.453645,0.729952,18.846201,19.798764,-0.108727,0.729952


,modelo,segmento,condicion,lead_minutes,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE
0,GradientBoosting_ARX,2021,delta_positivo_alto,10,220,0.100354,0.280569,0.824367,12.839990,14.042780,-0.061992,0.824367
1,GradientBoosting_ARX,2021,delta_positivo_alto,20,220,0.140385,0.354601,0.719453,18.573215,20.892907,-0.087392,0.719453
2,GradientBoosting_ARX,2021,delta_positivo_alto,30,220,0.161364,0.420577,0.605345,21.623044,24.983475,-0.113359,0.605345
3,GradientBoosting_ARX,2025,delta_positivo_alto,10,24,0.447925,0.573569,0.152637,22.931308,28.164611,-0.447925,0.152637
4,GradientBoosting_ARX,2025,delta_positivo_alto,20,24,0.700105,0.760451,-0.489501,36.030041,45.415864,-0.700105,-0.489501
5,GradientBoosting_ARX,2025,delta_positivo_alto,30,24,0.885022,0.962067,-1.384019,44.622541,58.734763,-0.868977,-1.384019
6,GradientBoosting_ARX,global,delta_positivo_alto,10,265,0.143489,0.323942,0.805504,12.626374,14.094729,-0.106099,0.805504
7,GradientBoosting_ARX,global,delta_positivo_alto,20,265,0.210562,0.418460,0.675449,19.016434,21.892558,-0.159731,0.675449
8,GradientBoosting_ARX,global,delta_positivo_alto,30,265,0.250507,0.504576,0.528124,22.108291,26.265612,-0.203213,0.528124
9,GradientBoosting_ARX,2021,lluvia_1h_alta,10,89,0.326222,0.570466,0.709874,22.885757,24.089025,-0.197390,0.709874


Metricas por condicion guardadas en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_metricas_condiciones.csv
Metricas por condicion y lead guardadas en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_metricas_condiciones_por_lead.csv


## Tablas resumen preparadas para comparacion posterior

## Tablas resumen

Ademas de las tablas completas, se generan tres tablas compactas:

- tabla global;
- tabla global a t+30;
- tabla de condiciones criticas a t+30.

Estas tablas no comparan contra otros modelos todavia. Su objetivo es dejar una salida equivalente a las tablas principales de otros notebooks, para que un notebook posterior pueda unirlas y construir una comparacion general.

In [15]:
global_table = metrics_df[metrics_df["segmento"] == "global"].copy().sort_values(["RMSE", "modelo"]).reset_index(drop=True)
t30_table = lead_metrics_df[
    (lead_metrics_df["segmento"] == "global")
    & (lead_metrics_df["lead_minutes"] == 30)
].copy().sort_values(["RMSE", "modelo"]).reset_index(drop=True)

conditions_for_paper = ["precipitacion_gt_0", "lluvia_1h_alta", "nivel_p90", "delta_positivo_alto"]
condition_t30_table = condition_lead_metrics_df[
    (condition_lead_metrics_df["segmento"] == "global")
    & (condition_lead_metrics_df["lead_minutes"] == 30)
    & (condition_lead_metrics_df["condicion"].isin(conditions_for_paper))
].copy().sort_values(["condicion", "RMSE", "modelo"]).reset_index(drop=True)

global_table_path = os.path.join(REPORTS_DIR, "23_tabla_principal_global.csv")
t30_table_path = os.path.join(REPORTS_DIR, "23_tabla_principal_t30.csv")
condition_t30_table_path = os.path.join(REPORTS_DIR, "23_tabla_condiciones_t30.csv")

global_table.to_csv(global_table_path, index=False)
t30_table.to_csv(t30_table_path, index=False)
condition_t30_table.to_csv(condition_t30_table_path, index=False)

display(global_table)
display(t30_table)
display(condition_t30_table)

print("Tabla global guardada en:", global_table_path)
print("Tabla t+30 guardada en:", t30_table_path)
print("Tabla condiciones t+30 guardada en:", condition_t30_table_path)

,modelo,segmento,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE
0,GradientBoosting_ARX,global,32064,0.015587,0.082917,0.924152,3.535511,3.575984,-0.008438,0.924152


,modelo,segmento,lead_minutes,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE
0,GradientBoosting_ARX,global,30,10688,0.019653,0.097683,0.894733,4.47014,4.531591,-0.011481,0.894733


,modelo,segmento,condicion,lead_minutes,n,MAE,RMSE,R2,MAPE (%),sMAPE (%),Bias,NSE
0,GradientBoosting_ARX,global,delta_positivo_alto,30,265,0.250507,0.504576,0.528124,22.108291,26.265612,-0.203213,0.528124
1,GradientBoosting_ARX,global,lluvia_1h_alta,30,141,0.521682,0.759887,0.457198,35.861515,39.080550,-0.400944,0.457198
2,GradientBoosting_ARX,global,nivel_p90,30,1403,0.090880,0.263558,0.677728,7.311592,8.149327,-0.070684,0.677728
3,GradientBoosting_ARX,global,precipitacion_gt_0,30,273,0.273200,0.528971,0.656160,22.312859,24.661277,-0.202577,0.656160


Tabla global guardada en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_tabla_principal_global.csv
Tabla t+30 guardada en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_tabla_principal_t30.csv
Tabla condiciones t+30 guardada en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_tabla_condiciones_t30.csv


## Importancia de variables

## Interpretacion de importancias

Gradient Boosting permite calcular `feature_importances_`, una medida basada en cuanto contribuye cada variable a reducir el error dentro de los arboles.

Como se usa `MultiOutputRegressor`, hay un estimador por horizonte. Por eso se reportan importancias separadas para:

```text
t+10
t+20
t+30
```

Esto puede ayudar a responder preguntas hidrologicas como:

- el nivel actual domina todos los horizontes?
- la lluvia acumulada se vuelve mas importante a t+30?
- los deltas recientes del nivel explican mejor los ascensos?
- hay diferencias entre variables relevantes para corto y mediano plazo?

Hay que interpretar estas importancias con cuidado: no son causalidad. Son una descripcion interna del modelo entrenado. Aun asi, son utiles para entender si el modelo esta usando variables hidrologicamente razonables.

In [ ]:
importance_rows = []

for horizon_idx, estimator in enumerate(final_model.estimators_, start=1):
    importances = getattr(estimator, "feature_importances_", None)
    if importances is None:
        continue

    for feature_name, importance in zip(FEATURE_NAMES, importances):
        importance_rows.append({
            "modelo": MODEL_NAME,
            "lead_step": horizon_idx,
            "lead_minutes": horizon_idx * 10,
            "feature": feature_name,
            "importance": float(importance),
        })

feature_importance_df = pd.DataFrame(importance_rows)
if not feature_importance_df.empty:
    feature_importance_df = feature_importance_df.sort_values(
        ["lead_minutes", "importance"],
        ascending=[True, False],
    ).reset_index(drop=True)

feature_importance_path = os.path.join(REPORTS_DIR, "23_feature_importances.csv")
feature_importance_df.to_csv(feature_importance_path, index=False)

display(feature_importance_df)
print("Importancias de variables guardadas en:", feature_importance_path)

,modelo,lead_step,lead_minutes,feature,importance
0,GradientBoosting_ARX,1,10,nivel_t,0.989193
1,GradientBoosting_ARX,1,10,delta_10m,0.003945
2,GradientBoosting_ARX,1,10,lluvia_60m,0.002287
3,GradientBoosting_ARX,1,10,delta_30m,0.001308
4,GradientBoosting_ARX,1,10,lluvia_max_60m,0.000953
5,GradientBoosting_ARX,1,10,nivel_media_30m,0.000690
6,GradientBoosting_ARX,1,10,delta_20m,0.000632
7,GradientBoosting_ARX,1,10,lluvia_90m,0.000242
8,GradientBoosting_ARX,1,10,nivel_media_90m,0.000197
9,GradientBoosting_ARX,1,10,nivel_media_60m,0.000156


Importancias de variables guardadas en: g:\Mi unidad\Maestria - IA\Tesis\Proyectos\Mburicao_Iberamia\reports\23_gradient_boosting_arx\23_feature_importances.csv


: 

## Salidas generadas

Este notebook genera las siguientes tablas en `reports/23_gradient_boosting_arx`:

- `23_dataset_resumen.csv`
- `23_split_resumen.csv`
- `23_dataset_supervisado_resumen.csv`
- `23_gradient_boosting_arx_gridsearch_metricas.csv`
- `23_gradient_boosting_arx_gridsearch_resumen_corridas.csv`
- `23_gradient_boosting_arx_gridsearch_mejor_config.csv`
- `23_predicciones_test_long.csv`
- `23_metricas_global_segmento.csv`
- `23_metricas_por_lead.csv`
- `23_metricas_condiciones.csv`
- `23_metricas_condiciones_por_lead.csv`
- `23_tabla_principal_global.csv`
- `23_tabla_principal_t30.csv`
- `23_tabla_condiciones_t30.csv`
- `23_feature_importances.csv`

El modelo entrenado se guarda en `models/23_gradient_boosting_arx/23_gradient_boosting_arx.joblib`.

Estas salidas quedan listas para ser usadas posteriormente en un notebook de comparacion global contra Ridge-ARX, N-BEATS, ARIMA, SARIMAX y LSTM.